# Real Estate Price Prediction: Feature Scaling and Model Comparison
Predict **house price per unit area** and compare how feature scaling affects different models.

**Data:** put `Real estate.csv` (Real Estate Valuation dataset) in this folder before running.

**What this notebook shows**
- Scaling does not change Linear Regression, but it matters for distance-based models like SVR.
- Random Forest and Gradient Boosting are compared with Linear Regression using R², MAE, and RMSE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load the data

In [ ]:
df = pd.read_csv("Real estate.csv")   # relative path, works on any computer
df = df.drop(columns=["No"])          # row number, not a real feature

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
df.head()

In [ ]:
TARGET = "Y house price of unit area"
X = df.drop(columns=[TARGET])
y = df[TARGET]

df.describe().T

## 2. Check feature ranges
The features sit on very different scales (for example distance to MRT is in the hundreds or thousands, while stores is 0 to 10). This is why scaling is tested.

In [ ]:
fig, axes = plt.subplots(1, X.shape[1], figsize=(16, 3))
for ax, col in zip(axes, X.columns):
    ax.boxplot(X[col])
    ax.set_title(col.split(" ", 1)[1][:18], fontsize=8)
plt.tight_layout()
plt.show()

## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), "train rows,", len(X_test), "test rows")

## 4. Compare models
Each scaler sits inside a pipeline, so it is fitted on training data only (no data leakage). SVR also gets a scaled target, because its default settings work poorly when prices are on a large scale.

In [ ]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    cv = cross_val_score(model, X_train, y_train, cv=KFold(5, shuffle=True, random_state=42), scoring="r2")
    return {
        "Model": name,
        "Test R2": r2_score(y_test, pred),
        "CV R2 (mean)": cv.mean(),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    }

models = {
    "Linear Regression (no scaling)": LinearRegression(),
    "Linear Regression + MinMax": make_pipeline(MinMaxScaler(), LinearRegression()),
    "Linear Regression + Robust": make_pipeline(RobustScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR (unscaled)": SVR(kernel="rbf"),
    "SVR + StandardScaler (scaled target)": TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10)),
        transformer=StandardScaler()),
}

results = pd.DataFrame([evaluate(n, m) for n, m in models.items()]).set_index("Model")
results.sort_values("Test R2", ascending=False).round(4)

## 5. Best model: actual vs predicted and feature importance

In [ ]:
best_name = results["Test R2"].idxmax()
print("Best model on test R2:", best_name)
best = models[best_name]
pred = best.predict(X_test)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=0.7)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual price"); plt.ylabel("Predicted price")
plt.title(f"Actual vs predicted: {best_name}")
plt.show()

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_train, y_train)
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
importance.plot(kind="barh", figsize=(7, 3.5), title="Random Forest feature importance")
plt.tight_layout()
plt.show()